# Kaggle: sanidade na GPU e tempo por execução

Notebook do item 6 de `docs/plano-reformulacao.md` e da pendência "medir o tempo na T4".

**Antes de rodar:** em *Settings*, ative **Accelerator → GPU T4 x2** (ou P100) e **Internet → On**
(exige conta verificada por telefone). O notebook clona o repositório público; as mudanças locais
precisam estar no GitHub (`BRANCH` abaixo) para entrarem aqui.

Etapas:
1. Ambiente (GPU, dependências).
2. **Sanidade na GPU:** FedAvg × FedProx μ = 0 (α = 0,1, semente 0, T = 20) devem ser idênticos rodada a rodada.
3. **Tempo:** uma execução do bloco A (FedAvg, α = 0,1, semente 0, T = 100). Na CPU Ryzen levou ~14 min.
4. Compacta os resultados gerados aqui em `/kaggle/working/resultados_kaggle.zip` (aba *Output*).

Os resultados versionados no repositório são movidos para `results_repo/` antes dos testes, para que o
zip contenha **só** o que foi gerado no Kaggle e não sobrescreva nada ao ser trazido de volta.
Lembrete: resultados não são bit a bit iguais entre máquinas; pares FedAvg × FedProx de uma semente
devem rodar inteiros na mesma máquina (`grid.py --shard` divide por semente).

In [ ]:
REPO = "https://github.com/el-yawd/seminar-iv.git"
BRANCH = "main"
!nvidia-smi -L
!rm -rf /kaggle/working/seminar-iv && git clone -q --depth 1 -b {BRANCH} {REPO} /kaggle/working/seminar-iv
%cd /kaggle/working/seminar-iv/experimentos
!git log -1 --format='%h %ad %s' --date=short

In [ ]:
!pip install -q -r requirements-gpu.txt
import torch, flwr, ray
print(torch.__version__, torch.version.cuda, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("flwr", flwr.__version__, "ray", ray.__version__)

In [ ]:
# separa os resultados versionados: o que for gerado daqui em diante fica sozinho em results/
!mv results results_repo && mkdir results

## 2. Sanidade na GPU (μ = 0 ≡ FedAvg)

In [ ]:
!python run.py --dataset fmnist --alpha 0.1 --strategy fedavg  --seed 0 --rounds 20 2>&1 | tail -1
!python run.py --dataset fmnist --alpha 0.1 --strategy fedprox --mu 0 --seed 0 --rounds 20 2>&1 | tail -1

In [ ]:
import pandas as pd
a = pd.read_csv("results/fmnist_dirichlet_a0.1_fedavg_E2_s0_T20.csv")
b = pd.read_csv("results/fmnist_dirichlet_a0.1_fedprox_mu0_E2_s0_T20.csv")
same_pids = (a.pids.fillna("") == b.pids.fillna("")).all()
max_dacc, max_dloss = (a.acc - b.acc).abs().max(), (a.loss - b.loss).abs().max()
print(f"mesmos clientes: {same_pids}; max |Δacc| = {max_dacc:.2e}; max |Δloss| = {max_dloss:.2e}")
print("IDÊNTICO na GPU" if same_pids and max_dacc == 0 and max_dloss == 0 else "NÃO idêntico: rodar pares em CPU ou documentar a tolerância")
print("tempo médio de rodada (s):", round(a.t_round.iloc[2:].mean(), 2))

## 3. Tempo de uma execução do bloco A (T = 100)

In [ ]:
!python run.py --dataset fmnist --alpha 0.1 --strategy fedavg --seed 0 --rounds 100 2>&1 | tail -1

In [ ]:
import json
js = json.load(open("results/fmnist_dirichlet_a0.1_fedavg_E2_s0_T100.json"))
c = pd.read_csv("results/fmnist_dirichlet_a0.1_fedavg_E2_s0_T100.csv")
t = js["wall_time_s"] / 60
print(f"tempo total: {t:.1f} min (Ryzen 7 3700U: ~14 min); rodada média: {c.t_round.iloc[2:].mean():.2f} s")
print(js["env"])
print(f"estimativa: bloco A restante, sementes 2-4 (60 execuções) ≈ {60 * t / 60:.1f} h; bloco D (100) ≈ {100 * t / 60:.1f} h")

## (Opcional) Rodar parte de um bloco

Escolha sementes que **nenhuma outra máquina** esteja rodando. Exemplo: bloco A, sementes 3 e 4
(`--shard` divide por semente: com `0/5`…`4/5`, cada fatia é uma semente). Sessões têm limite de 12 h; o
`grid.py` é retomável.

In [ ]:
# !python grid.py --block A --shard 3/5
# !python grid.py --block A --shard 4/5

## 4. Saída

In [ ]:
!cd /kaggle/working/seminar-iv/experimentos && zip -qr /kaggle/working/resultados_kaggle.zip results && ls -la /kaggle/working/resultados_kaggle.zip